# AIC 2026 - Kaggle frame extraction smoke

Attach these datasets before running:

- `lyduchoang/aic-26-video` for raw videos
- `khoalequangminh/aic-test-dataset` for metadata/map files

This notebook clones/pulls the repo branch and runs the offline frame extraction CLI for one video. TransNetV2 is included as an optional disabled cell for log/debug only; it does not run DAM, OCR, ASR, embeddings, or tests.

## Optional fast paths

If you already know the Kaggle paths, set these variables in the environment/setup cell to avoid scanning the full `/kaggle/input` tree:

- `AIC_VIDEO_PATH`: raw video file for `L21_V001`.
- `AIC_MAP_CSV`: optional organizer map CSV. If empty, smoke uses fallback timestamps.
- `AIC_MEDIA_INFO`: optional media-info JSON.
- `AIC_DISCOVER_SUPPORT_FILES=1`: only use this if you provide `AIC_VIDEO_PATH` but still want the notebook to scan for map/media files.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
BRANCH = "feat/offline-frame-extraction-transnetv2"
TARGET = Path("/kaggle/working/AIC-2026")

def run(command, *, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

clone_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
except Exception:
    token = None

if token:
    clone_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {token}",
    })

if TARGET.exists() and not (TARGET / ".git").is_dir():
    raise RuntimeError(f"Target exists but is not a git repo: {TARGET}")
if not TARGET.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(TARGET)], env=clone_env)
else:
    run(["git", "fetch", "origin", BRANCH], cwd=TARGET, env=clone_env)
    run(["git", "switch", BRANCH], cwd=TARGET, env=clone_env)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET, env=clone_env)

os.chdir(TARGET)
print("Repo:", Path.cwd())
run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=TARGET)
run(["git", "rev-parse", "--short", "HEAD"], cwd=TARGET)


In [ ]:
from collections import Counter
from pathlib import Path
import os
import subprocess

os.environ["AIC_DATA_ROOT"] = "/kaggle/input"
os.environ["AIC_ARTIFACT_ROOT"] = "/kaggle/working/aic2026-artifacts"
os.environ["AIC_VIDEO_ID"] = "L21_V001"
os.environ.setdefault("AIC_VIDEO_PATH", "")
os.environ.setdefault("AIC_MAP_CSV", "")
os.environ.setdefault("AIC_MEDIA_INFO", "")
os.environ.setdefault("AIC_DISCOVER_SUPPORT_FILES", "0")

def log(*args):
    print(*args, flush=True)

input_root = Path(os.environ["AIC_DATA_ROOT"])
explicit_video_path = os.environ.get("AIC_VIDEO_PATH", "").strip()
explicit_map_csv = os.environ.get("AIC_MAP_CSV", "").strip()
explicit_media_info = os.environ.get("AIC_MEDIA_INFO", "").strip()
discover_support_files = os.environ.get("AIC_DISCOVER_SUPPORT_FILES", "0") == "1"
log("AIC_DATA_ROOT:", input_root)
log("AIC_ARTIFACT_ROOT:", os.environ["AIC_ARTIFACT_ROOT"])
log("AIC_VIDEO_ID:", os.environ["AIC_VIDEO_ID"])
log("AIC_VIDEO_PATH:", explicit_video_path or "<auto-discover>")
log("AIC_MAP_CSV:", explicit_map_csv or "<auto/fallback>")
log("AIC_MEDIA_INFO:", explicit_media_info or "<auto/none>")
log("AIC_DISCOVER_SUPPORT_FILES:", discover_support_files)

log("\nGPU preflight:")
try:
    gpu_report = subprocess.run(
        ["nvidia-smi", "-L"],
        check=False,
        capture_output=True,
        text=True,
    )
    log(gpu_report.stdout.strip() or gpu_report.stderr.strip() or "nvidia-smi returned no output")
    if gpu_report.returncode == 0:
        gpu_lines = [line for line in gpu_report.stdout.splitlines() if line.strip()]
        t4_lines = [line for line in gpu_lines if "T4" in line.upper()]
        if len(t4_lines) != 2:
            log(f"WARNING: expected Kaggle GPU T4 x2, detected {len(t4_lines)} T4 GPU(s).")
except FileNotFoundError:
    log("WARNING: nvidia-smi is not available. Check Kaggle Settings > Accelerator = GPU T4 x2.")

def print_tree(root: Path, *, max_depth: int = 3, max_items: int = 200):
    log("\nInput tree preview:")
    shown = 0

    def walk(directory: Path, depth: int):
        nonlocal shown
        if shown >= max_items or depth > max_depth:
            return
        try:
            children = sorted(directory.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))
        except PermissionError as error:
            log("  " * (depth - 1) + f"<permission denied: {error}>")
            return
        for path in children:
            if shown >= max_items:
                return
            rel = path.relative_to(root)
            indent = "  " * (len(rel.parts) - 1)
            suffix = "/" if path.is_dir() else ""
            log(f"{indent}{rel.name}{suffix}")
            shown += 1
            if path.is_dir():
                walk(path, depth + 1)

    walk(root, 1)
    if shown >= max_items:
        log(f"... truncated after {max_items} items")

print_tree(input_root)

if explicit_video_path:
    log("\nExplicit AIC_VIDEO_PATH is set; skipping full /kaggle/input scan.")
    for label, raw_path in [
        ("video", explicit_video_path),
        ("map_csv", explicit_map_csv),
        ("media_info", explicit_media_info),
    ]:
        if raw_path:
            path = Path(raw_path)
            log(f"  {label}: {path} exists={path.exists()} is_file={path.is_file()}")
    if not explicit_map_csv:
        log("  map_csv not provided; smoke extraction will use fallback timestamps unless AIC_DISCOVER_SUPPORT_FILES=1.")
else:
    log("\nScanning /kaggle/input for suffix counts and first video candidates...")
    suffix_counts = Counter()
    video_suffixes = {".avi", ".mkv", ".mov", ".mp4", ".webm"}
    video_candidates = []
    scanned_files = 0
    for path in input_root.rglob("*"):
        if not path.is_file():
            continue
        scanned_files += 1
        suffix = path.suffix.lower() or "<none>"
        suffix_counts[suffix] += 1
        if suffix in video_suffixes and len(video_candidates) < 20:
            video_candidates.append(path)
        if scanned_files % 1000 == 0:
            log(f"  scanned_files={scanned_files}")

    log(f"Scan complete. files={scanned_files}")
    log("Suffix counts:")
    for suffix, count in suffix_counts.most_common(20):
        log(f"  {suffix}: {count}")

    log("First video candidates:")
    for path in video_candidates:
        log(" ", path)


In [ ]:
import json
import os
import subprocess
from pathlib import Path

artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
video_id = os.environ["AIC_VIDEO_ID"]
explicit_video_path = os.environ.get("AIC_VIDEO_PATH", "").strip()
explicit_map_csv = os.environ.get("AIC_MAP_CSV", "").strip()
explicit_media_info = os.environ.get("AIC_MEDIA_INFO", "").strip()
discover_support_files = os.environ.get("AIC_DISCOVER_SUPPORT_FILES", "0") == "1"
command = [
    "python",
    "scripts/extract_frame_samples.py",
    "--config",
    "configs/offline/frame_extraction.yaml",
    "--video-id",
    video_id,
    "--output-root",
    str(artifact_root),
    "--limit",
    "10",
    "--resume",
]
if explicit_video_path:
    command.extend(["--video-path", explicit_video_path])
    if discover_support_files:
        command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
else:
    command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
if explicit_map_csv:
    command.extend(["--map-csv", explicit_map_csv])
if explicit_media_info:
    command.extend(["--media-info", explicit_media_info])
print("$", " ".join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
lines = []
for line in process.stdout:
    print(line, end="", flush=True)
    lines.append(line.strip())
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Frame extraction CLI failed with code {return_code}")

report = None
for line in reversed([line for line in lines if line]):
    try:
        report = json.loads(line)
        break
    except json.JSONDecodeError:
        pass
if report is None:
    raise RuntimeError("Frame extraction CLI did not print a JSON report")
manifest_path = Path(report["output"])
print("Manifest:", manifest_path)
print("Frames:", report.get("frames"))


## Optional TransNetV2 shot detection

This cell is disabled by default. To run it on Kaggle, attach a TransNetV2 runtime/weights dataset and set either:

- `RUN_TRANSNETV2_MANUAL = True` in the next cell and `TRANSNETV2_ENTRYPOINT` to the inference script, or
- `AIC_RUN_TRANSNETV2=1` plus `AIC_TRANSNETV2_ENTRYPOINT`/`AIC_TRANSNETV2_WEIGHTS` environment variables.

Logs are streamed live from the CLI and TransNetV2 process.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

# Manual Kaggle toggle: change to True only after attaching TransNetV2 runtime/weights.
RUN_TRANSNETV2_MANUAL = False
RUN_TRANSNETV2 = RUN_TRANSNETV2_MANUAL or os.environ.get("AIC_RUN_TRANSNETV2", "0") == "1"
TRANSNETV2_ENTRYPOINT = Path(os.environ.get(
    "AIC_TRANSNETV2_ENTRYPOINT",
    "/kaggle/input/transnetv2/inference/transnetv2.py",
))
TRANSNETV2_WEIGHTS = os.environ.get("AIC_TRANSNETV2_WEIGHTS")
TRANSNETV2_SCENES_FILE = os.environ.get("AIC_TRANSNETV2_SCENES_FILE")
explicit_video_path = os.environ.get("AIC_VIDEO_PATH", "").strip()
discover_support_files = os.environ.get("AIC_DISCOVER_SUPPORT_FILES", "0") == "1"

print("RUN_TRANSNETV2:", RUN_TRANSNETV2, flush=True)
print("TRANSNETV2_ENTRYPOINT:", TRANSNETV2_ENTRYPOINT, flush=True)
print("TRANSNETV2_WEIGHTS:", TRANSNETV2_WEIGHTS, flush=True)
print("TRANSNETV2_SCENES_FILE:", TRANSNETV2_SCENES_FILE, flush=True)

if not RUN_TRANSNETV2:
    print("Skipping TransNetV2. Set RUN_TRANSNETV2_MANUAL=True or AIC_RUN_TRANSNETV2=1 to enable.", flush=True)
else:
    if not TRANSNETV2_SCENES_FILE and not TRANSNETV2_ENTRYPOINT.exists():
        raise FileNotFoundError(
            "TransNetV2 entrypoint does not exist. Attach a dataset containing TransNetV2 "
            "or set AIC_TRANSNETV2_ENTRYPOINT to the correct path."
        )
    command = [
        "python",
        "scripts/run_transnetv2_shots.py",
        "--config",
        "configs/offline/frame_extraction.yaml",
        "--video-id",
        os.environ["AIC_VIDEO_ID"],
        "--output-root",
        os.environ["AIC_ARTIFACT_ROOT"],
        "--resume",
    ]
    if explicit_video_path:
        command.extend(["--video-path", explicit_video_path])
        if discover_support_files:
            command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
    else:
        command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
    if TRANSNETV2_SCENES_FILE:
        command.extend(["--scenes-file", TRANSNETV2_SCENES_FILE])
    else:
        command.extend(["--entrypoint", str(TRANSNETV2_ENTRYPOINT)])
        if TRANSNETV2_WEIGHTS:
            command.extend(["--weights", TRANSNETV2_WEIGHTS])

    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line.strip())
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"TransNetV2 shot detection failed with code {return_code}")

    shot_report = None
    for line in reversed([line for line in lines if line]):
        try:
            shot_report = json.loads(line)
            break
        except json.JSONDecodeError:
            pass
    print("Shot report:", shot_report)


In [ ]:
import json
import os
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import Markdown, display

artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
manifest_path = artifact_root / "frame_extraction" / "manifests" / f"{os.environ['AIC_VIDEO_ID']}.jsonl"
records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]

def timecode(seconds: float) -> str:
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    total_m = total_s // 60
    m = total_m % 60
    h = total_m // 60
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

print("Records:", len(records))
print("Manifest:", manifest_path)
print("\nFull frame table:")
print("sample_n\ttimecode\tpts_time_s\tframe_idx\tfps\tkeyframe_n\tsource\timage_path")
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    print(
        f"{record['sample_n']}\t{timecode(record['pts_time_s'])}\t"
        f"{record['pts_time_s']:.6f}\t{record['frame_idx']}\t{record['fps']:.4f}\t"
        f"{record.get('keyframe_n')}\t{record['sampling_source']}\t{image_path}"
    )

thumbs = []
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    image = Image.open(image_path).convert("RGB")
    thumb = image.copy()
    thumb.thumbnail((240, 135))
    canvas = Image.new("RGB", (240, 160), "white")
    canvas.paste(thumb, ((240 - thumb.width) // 2, 0))
    draw = ImageDraw.Draw(canvas)
    draw.text(
        (8, 140),
        f"#{record['sample_n']} {timecode(record['pts_time_s'])}",
        fill=(0, 0, 0),
    )
    thumbs.append(canvas)

cols = 2
rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new("RGB", (cols * 240, max(1, rows) * 160), "white")
for index, thumb in enumerate(thumbs):
    sheet.paste(thumb, ((index % cols) * 240, (index // cols) * 160))
display(Markdown("## Thumbnail overview"))
display(sheet)

display(Markdown("## Full extracted frames"))
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    display(Markdown(
        f"### Frame #{record['sample_n']} - {timecode(record['pts_time_s'])} "
        f"({record['pts_time_s']:.3f}s, frame_idx={record['frame_idx']})"
    ))
    display(Image.open(image_path).convert("RGB"))
